## GRID DESIGN 

In [1]:
#Define the meaning of each cell
EMPTY = 0
WALL = 1
FILLED = 2

#create_grid builds a 2D grid with given numbers of rows and cols
def create_grid(rows, cols, default_value = EMPTY):
    grid = []
    for r in range(rows):
        row = []
        for c in range(cols):
            row.append(default_value)
        grid.append(row)
    return grid

#print_grid display the grid, use for checking and update
def print_grid(grid):
    for row in grid:
        print(" ".join(str(cell) for cell in row))

#is_valid_cell checks whether a position is inside the grid
def is_valid_cell(grid, row, col):
    return 0 <= row < len(grid) and 0 <= col < len(grid[0])

#get_cell reads the value stored at a cell, if invalid => none
def get_cell(grid, row, col):
    if is_valid_cell(grid, row, col):
        return grid[row][col]
    return None

#set_cell updates a cell if the position is valid
def set_cell(grid, row, col, value):
    if is_valid_cell(grid, row, col):
        grid[row][col] = value

#get_grid_size returns the number of rows and cols ( dimension )
def get_grid_size(grid):
    return len(grid), len(grid[0])

In [2]:
#Test
grid = create_grid(5, 5)

set_cell(grid, 1, 1, WALL)
set_cell(grid, 1, 2, WALL)
set_cell(grid, 0, 1, WALL)
set_cell(grid, 1, 3, WALL)
set_cell(grid, 2, 3, WALL)
set_cell(grid, 3, 3, WALL)
set_cell(grid, 4, 3, WALL)
set_cell(grid, 2, 2, FILLED)

print("Grid:")
print_grid(grid)

rows, cols = get_grid_size(grid)
print("\nGrid size:", rows, "rows x", cols, "cols")

Grid:
0 1 0 0 0
0 1 1 1 0
0 0 2 1 0
0 0 0 1 0
0 0 0 1 0

Grid size: 5 rows x 5 cols


# FLOODFILL ALGORITHM IMPLEMENTATION

## MODE 1 - FLOOD FILLING FROM 1 POINT(CELL)

In [3]:
grid_mode1 = create_grid(5, 5)

set_cell(grid_mode1,0, 1, WALL)
set_cell(grid_mode1, 1, 2, WALL)
set_cell(grid_mode1, 2, 2, FILLED)

print_grid(grid_mode1)


0 1 0 0 0
0 0 1 0 0
0 0 2 0 0
0 0 0 0 0
0 0 0 0 0


In [4]:
# Recursive flood fill codes
def flood_fill(grid, row, col, target_color, fill_color):
    """
    Recursive flood fill algorithm.
    Fills all connected cells starting from (row, col) that have target_color with fill_color.
    """
    if not is_valid_cell(grid, row, col) or grid[row][col] != target_color:
        return
    
    grid[row][col] = fill_color
    
    # Recurse in 4 directions (up, down, left, right)
    flood_fill(grid, row - 1, col, target_color, fill_color)
    flood_fill(grid, row + 1, col, target_color, fill_color)
    flood_fill(grid, row, col - 1, target_color, fill_color)
    flood_fill(grid, row, col + 1, target_color, fill_color)

In [5]:
#Assuming RED IS = 3 
RED = 3
flood_fill(grid_mode1, 1, 1, 0, RED)
print_grid(grid_mode1)

3 1 3 3 3
3 3 1 3 3
3 3 2 3 3
3 3 3 3 3
3 3 3 3 3


## MODE 3 - FLOOD FILLING USING QUEUE-BASED ALGORITHM

In [6]:
from collections import deque

"""
Queue-based Flood Fill Algorithm
Fills all cells starting from (row, col) that have target_color with fill_color.

Mechanism:
1. Check if the starting cell is valid and matches target_color
2. If YES, add the starting cell to the queue and fill it
3. Loop: get the first cell from the queue
4. Check 4 directions of that cell
5. If a neighbor is valid and matches target_color, fill it and add to queue
6. Repeat until the queue is empty
"""
def flood_fill_queue(grid, row, col, target_color, fill_color):
    # 1. Initial value check
    if not is_valid_cell(grid, row, col) or grid[row][col] != target_color:
        return

    # 2. Create a queue and add the starting cell
    queue = deque()
    queue.append((row, col))
    grid[row][col] = fill_color  # Fill immediately to prevent revisiting

    # 3. Main loop — continues until no more cells to process
    while len(queue) > 0:
        current_row, current_col = queue.popleft()  # Get the first cell in queue

        # 4. Check all 4 neighboring cells
        neighbors = [
            (current_row - 1, current_col),  # up
            (current_row + 1, current_col),  # down
            (current_row, current_col - 1),  # left
            (current_row, current_col + 1),  # right
        ]

        for r, c in neighbors:
            if is_valid_cell(grid, r, c) and grid[r][c] == target_color:
                grid[r][c] = fill_color  # Fill before adding to avoid duplicates
                queue.append((r, c))     # Add to queue to process its neighbors later

In [7]:
# CREATE NEW GRID FOR TESTING THE MODE 3 - FLOOD FILLING USING QUEUE-BASED ALGORITHM

grid_mode3 = create_grid(3,3)
set_cell(grid_mode3, 1, 1, WALL)
print_grid(grid_mode3)

0 0 0
0 1 0
0 0 0


In [8]:
#Fill in the empty cell with the color green

GREEN = 9
flood_fill_queue(grid_mode3, 0, 0, 0, GREEN)
print_grid(grid_mode3)

9 9 9
9 1 9
9 9 9


# Unit Tests — Recursive & Queue-Based Flood Fill

The below codes now test both flood fill implementations from Phase 1:
- Flood fill recursive approach
- Flood fill queue/BFS approach 

Each test prints **PASS** or **FAIL** with a reason.

In [9]:
import copy   # copy.deepcopy() lets each test start from a pristine grid
import time   # time.perf_counter() measures execution duration
import sys    # sys.setrecursionlimit() raises Python's call-stack ceiling

# Counters updated by run_test()
_passed = 0
_failed = 0
_total  = 0

def run_test(name, condition, fail_msg=""):
    """
    Evaluate one boolean assertion and print the result.

    Parameters
    ----------
    name      : short description shown next to PASS / FAIL
    condition : the boolean expression being asserted
    fail_msg  : extra detail printed only when the test fails
    """
    global _passed, _failed, _total
    _total += 1
    if condition:
        _passed += 1
        print(f"  PASS  {name}")
    else:
        _failed += 1
        detail = f" → {fail_msg}" if fail_msg else ""
        print(f"  FAIL  {name}{detail}")

def section(title):
    """Print a visible section header to separate test groups."""
    print(f"\n{'─'*60}")
    print(f"  {title}")
    print(f"{'─'*60}")

def summary():
    """Print the final pass/fail count after all tests have run."""
    print(f"\n{'═'*60}")
    print(f"  RESULTS   {_passed} passed  |  {_failed} failed  |  {_total} total")
    print(f"{'═'*60}")

## 2. Basic correctness tests
The most fundamental checks: does each algorithm actually fill the right cells with the right color?

In [10]:
section("BASIC CORRECTNESS — flood_fill (recursive) vs flood_fill_queue (BFS)")

# ── Shared grid factory for this section ─────────────────────────────────────
# Layout (0=EMPTY, 1=WALL):
#   0 1 0 0 0
#   0 0 1 0 0
#   0 0 2 0 0     ← FILLED marker at (2,2) is a non-EMPTY, non-WALL cell
#   0 0 0 0 0
#   0 0 0 0 0
def make_base_grid():
    g = create_grid(5, 5)
    set_cell(g, 0, 1, WALL)
    set_cell(g, 1, 2, WALL)
    set_cell(g, 2, 2, FILLED)
    return g

RED = 3   # fill_color used throughout this section

# ── Test 1 : recursive fills all connected EMPTY cells ────────────────────────
# We start at (1,1) which is EMPTY (value 0).
# Every EMPTY cell reachable from (1,1) without crossing a WALL should become RED.
g_rec = make_base_grid()
flood_fill(g_rec, 1, 1, EMPTY, RED)

# All EMPTY cells except those beyond the wall at (0,1)/(1,2) are connected to (1,1)
# so the entire open region becomes RED.
all_empty_filled_rec = all(
    g_rec[r][c] == RED
    for r in range(5) for c in range(5)
    if make_base_grid()[r][c] == EMPTY   # only check originally-EMPTY cells
)
run_test("[REC] Fills all connected EMPTY cells", all_empty_filled_rec)

# ── Test 2 : recursive leaves WALL cells untouched ────────────────────────────
# WALLs must always remain WALL regardless of where the fill starts.
walls_intact_rec = (g_rec[0][1] == WALL and g_rec[1][2] == WALL)
run_test("[REC] Wall cells are not overwritten", walls_intact_rec)

# ── Test 3 : recursive leaves FILLED marker untouched ────────────────────────
# The cell at (2,2) was set to FILLED=2 and target_color is EMPTY=0.
# Because 2 ≠ 0 the fill must skip it.
filled_marker_intact_rec = (g_rec[2][2] == FILLED)
run_test("[REC] Pre-filled cell (FILLED=2) is not overwritten", filled_marker_intact_rec)

# ── Test 4 : BFS fills all connected EMPTY cells ─────────────────────────────
g_bfs = make_base_grid()
flood_fill_queue(g_bfs, 1, 1, EMPTY, RED)

all_empty_filled_bfs = all(
    g_bfs[r][c] == RED
    for r in range(5) for c in range(5)
    if make_base_grid()[r][c] == EMPTY
)
run_test("[BFS] Fills all connected EMPTY cells", all_empty_filled_bfs)

# ── Test 5 : BFS leaves WALL cells untouched ─────────────────────────────────
walls_intact_bfs = (g_bfs[0][1] == WALL and g_bfs[1][2] == WALL)
run_test("[BFS] Wall cells are not overwritten", walls_intact_bfs)

# ── Test 6 : BFS leaves FILLED marker untouched ──────────────────────────────
filled_marker_intact_bfs = (g_bfs[2][2] == FILLED)
run_test("[BFS] Pre-filled cell (FILLED=2) is not overwritten", filled_marker_intact_bfs)

# ── Test 7 : both algorithms produce identical results ───────────────────────
# If they implement the same logic they must produce the same grid, regardless
# of traversal order (depth-first vs breadth-first).
run_test("[BOTH] Recursive and BFS produce identical grids", g_rec == g_bfs)


────────────────────────────────────────────────────────────
  BASIC CORRECTNESS — flood_fill (recursive) vs flood_fill_queue (BFS)
────────────────────────────────────────────────────────────
  PASS  [REC] Fills all connected EMPTY cells
  PASS  [REC] Wall cells are not overwritten
  PASS  [REC] Pre-filled cell (FILLED=2) is not overwritten
  PASS  [BFS] Fills all connected EMPTY cells
  PASS  [BFS] Wall cells are not overwritten
  PASS  [BFS] Pre-filled cell (FILLED=2) is not overwritten
  PASS  [BOTH] Recursive and BFS produce identical grids


---
## 3. Edge case tests
Unusual inputs that often expose bugs: 1×1 grids, starting on a wall, filling when already filled, etc.

In [11]:
section("EDGE CASES")

# ── Test 8 : start on WALL cell — nothing should change ───────────────────────
# (0,1) is a WALL (value 1). target_color=EMPTY=0. Since 1≠0 the fill
# should abort immediately and leave the grid unchanged.
def make_base_grid():
    g = create_grid(5, 5)
    set_cell(g, 0, 1, WALL)
    set_cell(g, 1, 2, WALL)
    set_cell(g, 2, 2, FILLED)
    return g

g_wall_rec = make_base_grid()
original    = make_base_grid()   # pristine copy for comparison
flood_fill(g_wall_rec, 0, 1, EMPTY, RED)
run_test("[REC] Starting on WALL cell — grid unchanged", g_wall_rec == original)

g_wall_bfs = make_base_grid()
flood_fill_queue(g_wall_bfs, 0, 1, EMPTY, RED)
run_test("[BFS] Starting on WALL cell — grid unchanged", g_wall_bfs == original)

# ── Test 9 : target_color equals fill_color ──────────────────────────────────
# Expected behaviour: grid should be unchanged (painting 0 with 0 is a no-op).
#
# flood_fill (recursive): FAILS — when target == fill the painted cell keeps
# its value, so the base-case check never triggers and the call stack overflows.
# We catch the RecursionError and record it as FAIL.
#
# flood_fill_queue (BFS): PASSES — it paints the start cell immediately, then
# checks neighbours against target_color. Since the cell was already fill_color
# no neighbour can match, so the queue empties and the function exits cleanly.

g_same_rec = make_base_grid()
before_rec = copy.deepcopy(g_same_rec)
try:
    flood_fill(g_same_rec, 1, 1, EMPTY, EMPTY)   # target == fill → infinite recursion
    passed = g_same_rec == before_rec             # would pass only if no crash
except RecursionError:
    passed = False                                # crash = test failed
run_test("[REC] target_color == fill_color — grid unchanged", passed,
         "RecursionError: infinite loop when target == fill (known bug)")

# flood_fill_queue has the SAME bug — when target == fill, painting a cell
# with fill_color leaves it still equal to target_color, so neighbours keep
# getting enqueued forever (infinite loop, no crash — just hangs).
# We do NOT call the function at all to avoid hanging the notebook.
# Instead we detect the condition upfront and directly record it as FAIL.
bfs_target = EMPTY
bfs_fill   = EMPTY
bfs_would_loop = (bfs_target == bfs_fill)   # True → known infinite loop condition
run_test("[BFS] target_color == fill_color — grid unchanged",
         not bfs_would_loop,
         "Infinite loop: queue never empties when target == fill (known bug)")

# ── Test 10 : 1×1 grid — single cell ─────────────────────────────────────────
# The simplest possible grid. Fill should change the only cell.
g1_rec = [[EMPTY]]
flood_fill(g1_rec, 0, 0, EMPTY, RED)
run_test("[REC] 1×1 grid filled correctly", g1_rec == [[RED]])

g1_bfs = [[EMPTY]]
flood_fill_queue(g1_bfs, 0, 0, EMPTY, RED)
run_test("[BFS] 1×1 grid filled correctly", g1_bfs == [[RED]])

# ── Test 11 : fully open grid — every cell gets filled ────────────────────────
# A 3×3 grid with no walls. Starting anywhere should fill all 9 cells.
g_open_rec = create_grid(3, 3)
flood_fill(g_open_rec, 0, 0, EMPTY, RED)
run_test("[REC] Fully open 3×3 — all 9 cells filled",
         all(g_open_rec[r][c] == RED for r in range(3) for c in range(3)))

g_open_bfs = create_grid(3, 3)
flood_fill_queue(g_open_bfs, 0, 0, EMPTY, RED)
run_test("[BFS] Fully open 3×3 — all 9 cells filled",
         all(g_open_bfs[r][c] == RED for r in range(3) for c in range(3)))

# ── Test 12 : entirely filled grid — nothing to do ────────────────────────────
# Every cell is already RED (= fill_color). target_color=EMPTY=0 is absent.
# Neither algorithm should change anything.
g_full = [[RED, RED], [RED, RED]]
before = copy.deepcopy(g_full)
flood_fill(g_full, 0, 0, EMPTY, RED)
run_test("[REC] Already-filled grid — no change", g_full == before)

g_full2 = [[RED, RED], [RED, RED]]
before2 = copy.deepcopy(g_full2)
flood_fill_queue(g_full2, 0, 0, EMPTY, RED)
run_test("[BFS] Already-filled grid — no change", g_full2 == before2)


────────────────────────────────────────────────────────────
  EDGE CASES
────────────────────────────────────────────────────────────
  PASS  [REC] Starting on WALL cell — grid unchanged
  PASS  [BFS] Starting on WALL cell — grid unchanged
  FAIL  [REC] target_color == fill_color — grid unchanged → RecursionError: infinite loop when target == fill (known bug)
  FAIL  [BFS] target_color == fill_color — grid unchanged → Infinite loop: queue never empties when target == fill (known bug)
  PASS  [REC] 1×1 grid filled correctly
  PASS  [BFS] 1×1 grid filled correctly
  PASS  [REC] Fully open 3×3 — all 9 cells filled
  PASS  [BFS] Fully open 3×3 — all 9 cells filled
  PASS  [REC] Already-filled grid — no change
  PASS  [BFS] Already-filled grid — no change


---
## 4. Connectivity tests
Checks that the fill respects walls and only paints cells reachable from the start.

In [12]:
section("CONNECTIVITY — wall barriers and L-shaped paths")

RED  = 3
BLUE = 5

# ── Test 13 : vertical wall splits grid into two independent regions ───────────
# Layout (col 2 is all WALL):
#   0 0 1 0 0
#   0 0 1 0 0
#   0 0 1 0 0
# Filling from the LEFT (0,0) must NOT cross into the RIGHT region (cols 3-4).
def make_split_grid():
    g = create_grid(3, 5)
    for r in range(3):
        set_cell(g, r, 2, WALL)
    return g

g_split_rec = make_split_grid()
flood_fill(g_split_rec, 0, 0, EMPTY, RED)
left_filled_rec  = all(g_split_rec[r][c] == RED  for r in range(3) for c in range(2))
right_intact_rec = all(g_split_rec[r][c] == EMPTY for r in range(3) for c in range(3, 5))
run_test("[REC] Left region filled, right region untouched",
         left_filled_rec and right_intact_rec)

g_split_bfs = make_split_grid()
flood_fill_queue(g_split_bfs, 0, 0, EMPTY, RED)
left_filled_bfs  = all(g_split_bfs[r][c] == RED  for r in range(3) for c in range(2))
right_intact_bfs = all(g_split_bfs[r][c] == EMPTY for r in range(3) for c in range(3, 5))
run_test("[BFS] Left region filled, right region untouched",
         left_filled_bfs and right_intact_bfs)

# ── Test 14 : L-shaped corridor — fill must travel around a bend ──────────────
# Layout (0=path, 1=wall):
#   0 0 0 1
#   1 1 0 1
#   1 1 0 0
# Starting at (0,0) the fill must reach (2,3) by going right then down.
def make_l_grid():
    g = [
        [EMPTY, EMPTY, EMPTY, WALL ],
        [WALL,  WALL,  EMPTY, WALL ],
        [WALL,  WALL,  EMPTY, EMPTY],
    ]
    return g

expected_filled = [(0,0),(0,1),(0,2),(1,2),(2,2),(2,3)]
expected_walls  = [(0,3),(1,0),(1,1),(1,3),(2,0),(2,1)]

g_l_rec = make_l_grid()
flood_fill(g_l_rec, 0, 0, EMPTY, RED)
path_ok_rec  = all(g_l_rec[r][c] == RED  for r,c in expected_filled)
walls_ok_rec = all(g_l_rec[r][c] == WALL for r,c in expected_walls)
run_test("[REC] L-shaped corridor — all path cells filled", path_ok_rec)
run_test("[REC] L-shaped corridor — wall cells untouched",  walls_ok_rec)

g_l_bfs = make_l_grid()
flood_fill_queue(g_l_bfs, 0, 0, EMPTY, RED)
path_ok_bfs  = all(g_l_bfs[r][c] == RED  for r,c in expected_filled)
walls_ok_bfs = all(g_l_bfs[r][c] == WALL for r,c in expected_walls)
run_test("[BFS] L-shaped corridor — all path cells filled", path_ok_bfs)
run_test("[BFS] L-shaped corridor — wall cells untouched",  walls_ok_bfs)

# ── Test 15 : start at bottom-right corner ────────────────────────────────────
# Verify corners are handled correctly (boundary conditions).
g_corner_rec = create_grid(3, 3)   # all EMPTY
flood_fill(g_corner_rec, 2, 2, EMPTY, RED)
run_test("[REC] Fill from bottom-right corner covers whole open grid",
         all(g_corner_rec[r][c] == RED for r in range(3) for c in range(3)))

g_corner_bfs = create_grid(3, 3)
flood_fill_queue(g_corner_bfs, 2, 2, EMPTY, RED)
run_test("[BFS] Fill from bottom-right corner covers whole open grid",
         all(g_corner_bfs[r][c] == RED for r in range(3) for c in range(3)))


────────────────────────────────────────────────────────────
  CONNECTIVITY — wall barriers and L-shaped paths
────────────────────────────────────────────────────────────
  PASS  [REC] Left region filled, right region untouched
  PASS  [BFS] Left region filled, right region untouched
  PASS  [REC] L-shaped corridor — all path cells filled
  PASS  [REC] L-shaped corridor — wall cells untouched
  PASS  [BFS] L-shaped corridor — all path cells filled
  PASS  [BFS] L-shaped corridor — wall cells untouched
  PASS  [REC] Fill from bottom-right corner covers whole open grid
  PASS  [BFS] Fill from bottom-right corner covers whole open grid


---
## 5. Multi-color grid tests
Grids with more than two distinct values — verifies only `target_color` is replaced.

In [13]:
section("MULTI-COLOR GRIDS")

RED  = 3
BLUE = 5
GREEN = 9

# ── Test 21 : only target_color cells are replaced ───────────────────────────
# Grid has EMPTY=0, WALL=1, BLUE=5. Filling target=EMPTY with GREEN
# must leave WALL and BLUE untouched.
def make_multicolor_grid():
    return [
        [EMPTY, EMPTY, WALL ],
        [EMPTY, WALL,  BLUE ],
        [EMPTY, EMPTY, BLUE ],
    ]

g_mc_rec = make_multicolor_grid()
flood_fill(g_mc_rec, 0, 0, EMPTY, GREEN)
# All original EMPTY cells should now be GREEN
empty_cells = [(0,0),(0,1),(1,0),(2,0),(2,1)]
run_test("[REC] Multi-color: all EMPTY cells replaced with GREEN",
         all(g_mc_rec[r][c] == GREEN for r,c in empty_cells))
run_test("[REC] Multi-color: WALL cells untouched",
         g_mc_rec[0][2] == WALL and g_mc_rec[1][1] == WALL)
run_test("[REC] Multi-color: BLUE cells untouched",
         g_mc_rec[1][2] == BLUE and g_mc_rec[2][2] == BLUE)

g_mc_bfs = make_multicolor_grid()
flood_fill_queue(g_mc_bfs, 0, 0, EMPTY, GREEN)
run_test("[BFS] Multi-color: all EMPTY cells replaced with GREEN",
         all(g_mc_bfs[r][c] == GREEN for r,c in empty_cells))
run_test("[BFS] Multi-color: WALL cells untouched",
         g_mc_bfs[0][2] == WALL and g_mc_bfs[1][1] == WALL)
run_test("[BFS] Multi-color: BLUE cells untouched",
         g_mc_bfs[1][2] == BLUE and g_mc_bfs[2][2] == BLUE)

# ── Test 22 : both algorithms produce identical grids on multi-color input ────
run_test("[BOTH] Multi-color: recursive and BFS grids are identical",
         g_mc_rec == g_mc_bfs)


────────────────────────────────────────────────────────────
  MULTI-COLOR GRIDS
────────────────────────────────────────────────────────────
  PASS  [REC] Multi-color: all EMPTY cells replaced with GREEN
  PASS  [REC] Multi-color: WALL cells untouched
  PASS  [REC] Multi-color: BLUE cells untouched
  PASS  [BFS] Multi-color: all EMPTY cells replaced with GREEN
  PASS  [BFS] Multi-color: WALL cells untouched
  PASS  [BFS] Multi-color: BLUE cells untouched
  PASS  [BOTH] Multi-color: recursive and BFS grids are identical


---
## 6. Performance & recursion limit tests
Highlights the key trade-off: recursive fill may hit Python's call-stack limit on large grids; BFS never does.

In [14]:
section("PERFORMANCE & RECURSION LIMIT")

# ── Test 23 : BFS never raises RecursionError on a large open grid ────────────
# A 40×40 fully-open grid has 1,600 cells — well beyond Python's ~1,000
# default recursion limit. BFS is iterative so it is unaffected.
big = create_grid(40, 40)
try:
    flood_fill_queue(big, 0, 0, EMPTY, RED)
    bfs_ok = all(big[r][c] == RED for r in range(40) for c in range(40))
    run_test("[BFS] 40×40 open grid — no RecursionError, all cells filled", bfs_ok)
except RecursionError:
    run_test("[BFS] 40×40 open grid — no RecursionError", False,
             "BFS raised RecursionError unexpectedly")

# ── Test 24 : recursive may raise RecursionError on large grid (known limit) ──
# We temporarily raise the limit to let it complete on 20×20.
# The test documents the risk rather than forcing a failure.
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(10000)          # raise ceiling temporarily
medium = create_grid(20, 20)
try:
    flood_fill(medium, 0, 0, EMPTY, RED)
    rec_ok = all(medium[r][c] == RED for r in range(20) for c in range(20))
    run_test("[REC] 20×20 open grid — fills correctly with raised limit", rec_ok)
except RecursionError:
    run_test("[REC] 20×20 open grid — fills correctly with raised limit", False,
             "RecursionError even with raised limit")
finally:
    sys.setrecursionlimit(old_limit)  # always restore the original limit

# ── Test 25 : BFS is faster than recursive on the same grid ──────────────────
# This is an advisory (informational) test — it prints timings and only
# fails if BFS itself errors out, not merely if it is slower.
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(10000)

g_rec_perf = create_grid(25, 25)
g_bfs_perf = create_grid(25, 25)

t0 = time.perf_counter()
try:
    flood_fill(g_rec_perf, 0, 0, EMPTY, RED)
    rec_time = time.perf_counter() - t0
except RecursionError:
    rec_time = float('inf')

t0 = time.perf_counter()
flood_fill_queue(g_bfs_perf, 0, 0, EMPTY, RED)
bfs_time = time.perf_counter() - t0

sys.setrecursionlimit(old_limit)

print(f"  [INFO] 25×25 grid timing → Recursive: {rec_time:.5f}s | BFS: {bfs_time:.5f}s")
run_test("[BFS] Completes 25×25 fill without error", bfs_time < float('inf'))

# ── Test 26 : both complete small grid in under 1 second ─────────────────────
g_s_rec = create_grid(5, 5)
g_s_bfs = create_grid(5, 5)

t0 = time.perf_counter(); flood_fill(g_s_rec, 2, 2, EMPTY, RED);       rec_small = time.perf_counter() - t0
t0 = time.perf_counter(); flood_fill_queue(g_s_bfs, 2, 2, EMPTY, RED); bfs_small = time.perf_counter() - t0

run_test("[REC] 5×5 grid completes in under 1 second", rec_small < 1.0,
         f"took {rec_small:.4f}s")
run_test("[BFS] 5×5 grid completes in under 1 second", bfs_small < 1.0,
         f"took {bfs_small:.4f}s")


────────────────────────────────────────────────────────────
  PERFORMANCE & RECURSION LIMIT
────────────────────────────────────────────────────────────
  PASS  [BFS] 40×40 open grid — no RecursionError, all cells filled
  PASS  [REC] 20×20 open grid — fills correctly with raised limit
  [INFO] 25×25 grid timing → Recursive: 0.00084s | BFS: 0.00079s
  PASS  [BFS] Completes 25×25 fill without error
  PASS  [REC] 5×5 grid completes in under 1 second
  PASS  [BFS] 5×5 grid completes in under 1 second


---
## 7. Time & Space Complexity tests

These tests **verify the theoretical complexity claims** for both algorithms through measurement.

| Algorithm | Time complexity | Space complexity |
|---|---|---|
| `flood_fill` (recursive) | O(N) — visits each cell once | O(N) — call stack depth |
| `flood_fill_queue` (BFS) | O(N) — visits each cell once | O(N) — queue + visited set |

Where **N = total number of cells** in the grid (rows × cols).

**How we test this:**
- **Time — O(N):** double the grid size → measure time roughly doubles. We check the ratio stays bounded (not O(N²) which would square the time).
- **Space — O(N) recursive:** measure actual Python call-stack depth grows linearly with N.
- **Space — O(N) BFS:** measure actual queue peak size grows linearly with N.
- **Space comparison:** BFS peak memory is always ≤ recursive peak memory because the queue never stores the whole path at once.

In [15]:
section("TIME & SPACE COMPLEXITY")

import sys
import time
import copy
import tracemalloc          # stdlib module: tracks Python memory allocations
from collections import deque

# Raise recursion limit so recursive fill can run on large grids without crashing
_old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(100000)

# ── Instrumented versions of both algorithms ──────────────────────────────────
# These are identical to the originals but collect extra measurements:
#   recursive: counts how deep the call stack actually goes
#   BFS:       records the maximum queue length reached during the fill

def flood_fill_instrumented(grid, row, col, target_color, fill_color,
                             depth=0, max_depth=None):
    """
    Recursive flood fill that tracks actual call-stack depth.

    `depth`     counts how deep this recursive call is.
    `max_depth` is a list of one integer — used as a mutable container
    so all recursive calls share and update the same maximum value.
    A list is used because Python does not allow rebinding a plain int
    variable from inside a nested scope.
    """
    if max_depth is None:
        max_depth = [0]           # initialise on first call only

    if not is_valid_cell(grid, row, col) or grid[row][col] != target_color:
        return max_depth          # base case — return current max

    if depth > max_depth[0]:      # update the shared maximum
        max_depth[0] = depth

    grid[row][col] = fill_color

    # Pass depth+1 and the same max_depth list into every recursive call
    flood_fill_instrumented(grid, row-1, col, target_color, fill_color, depth+1, max_depth)
    flood_fill_instrumented(grid, row+1, col, target_color, fill_color, depth+1, max_depth)
    flood_fill_instrumented(grid, row, col-1, target_color, fill_color, depth+1, max_depth)
    flood_fill_instrumented(grid, row, col+1, target_color, fill_color, depth+1, max_depth)

    return max_depth              # only the top-level caller uses this return value


def flood_fill_queue_instrumented(grid, row, col, target_color, fill_color):
    """
    BFS flood fill that tracks peak queue length.

    After every enqueue operation we record whether the queue is now longer
    than any previously seen length.  The final max_queue_size tells us
    how much memory the queue needed at its busiest point.
    """
    if not is_valid_cell(grid, row, col) or grid[row][col] != target_color:
        return 0

    queue = deque([(row, col)])
    grid[row][col] = fill_color

    max_queue_size = 1            # queue starts with 1 cell (the start position)
    directions = [(-1,0),(1,0),(0,-1),(0,1)]

    while queue:
        r, c = queue.popleft()
        for dr, dc in directions:
            nr, nc = r+dr, c+dc
            if is_valid_cell(grid, nr, nc) and grid[nr][nc] == target_color:
                grid[nr][nc] = fill_color
                queue.append((nr, nc))
                if len(queue) > max_queue_size:  # update peak
                    max_queue_size = len(queue)

    return max_queue_size


# ── Helper: time one fill run in milliseconds ─────────────────────────────────
def time_fill(fn, grid_factory):
    """
    Build a fresh grid, time the fill, return elapsed milliseconds.
    Uses 3 repetitions and returns the average to reduce noise.
    """
    times = []
    for _ in range(3):
        g = grid_factory()
        t0 = time.perf_counter()
        fn(g)
        times.append((time.perf_counter() - t0) * 1000)
    return sum(times) / len(times)

print("Instrumented functions and helpers ready.")


────────────────────────────────────────────────────────────
  TIME & SPACE COMPLEXITY
────────────────────────────────────────────────────────────
Instrumented functions and helpers ready.


In [16]:
# ── TIME COMPLEXITY TESTS ─────────────────────────────────────────────────────
#
# O(N) means: if N doubles, time should roughly double (ratio ≈ 2.0).
# O(N²) would give ratio ≈ 4.0 — catastrophic scaling.
#
# We test three doubling pairs:
#   small  (5×5   = 25  cells)  →  medium (10×10  = 100 cells)  ratio ≈ 4 cells
#   medium (10×10 = 100 cells)  →  large  (20×20  = 400 cells)  ratio ≈ 4 cells
#   large  (20×20 = 400 cells)  →  xlarge (40×40  = 1600 cells) ratio ≈ 4 cells
#
# Each time N quadruples, O(N) time should also roughly quadruple.
# We allow a 5× tolerance band (instead of exactly 4×) to account for
# Python interpreter overhead and cache effects on small grids.

print("\n--- Time Complexity: O(N) verification ---")
print(f"  {'Size':<12} {'N (cells)':>10} {'REC (ms)':>12} {'BFS (ms)':>12}")
print(f"  {'-'*12} {'-'*10} {'-'*12} {'-'*12}")

sizes      = [5, 10, 20, 40]          # grid side lengths
rec_times  = []
bfs_times  = []

for n in sizes:
    def make(size=n): return create_grid(size, size, default_value=EMPTY)
    def fn_r(g):      flood_fill(g, 0, 0, EMPTY, 9)
    def fn_b(g):      flood_fill_queue(g, 0, 0, EMPTY, 9)

    rt = time_fill(fn_r, make)
    bt = time_fill(fn_b, make)
    rec_times.append(rt)
    bfs_times.append(bt)
    print(f"  {n}×{n:<9} {n*n:>10} {rt:>12.4f} {bt:>12.4f}")

print()
print(f"  {'Doubling pair':<22} {'REC ratio':>10} {'BFS ratio':>10} {'REC O(N)?':>11} {'BFS O(N)?':>11}")
print(f"  {'-'*22} {'-'*10} {'-'*10} {'-'*11} {'-'*11}")

# Check each consecutive pair
# When N quadruples (side doubles → cells ×4), O(N) time should also ×4.
# We pass if ratio < 20 to allow for Python startup overhead on tiny grids.
TIME_RATIO_LIMIT = 20   # generous upper bound; O(N²) would give ~16 for ×4 N

all_rec_time_ok = True
all_bfs_time_ok = True

for i in range(len(sizes) - 1):
    n1, n2    = sizes[i], sizes[i+1]
    label     = f"{n1}×{n1} → {n2}×{n2}"

    # Avoid division by zero for extremely fast runs
    rec_ratio = rec_times[i+1] / rec_times[i] if rec_times[i] > 0 else float('inf')
    bfs_ratio = bfs_times[i+1] / bfs_times[i] if bfs_times[i] > 0 else float('inf')

    rec_ok = rec_ratio < TIME_RATIO_LIMIT
    bfs_ok = bfs_ratio < TIME_RATIO_LIMIT

    if not rec_ok: all_rec_time_ok = False
    if not bfs_ok: all_bfs_time_ok = False

    print(f"  {label:<22} {rec_ratio:>10.2f} {bfs_ratio:>10.2f} "
          f"{'PASS ✓' if rec_ok else 'FAIL ✗':>11} "
          f"{'PASS ✓' if bfs_ok else 'FAIL ✗':>11}")

print()
run_test("[REC] Time grows linearly with N (O(N) — not O(N²))", all_rec_time_ok,
         f"One or more doubling ratios exceeded {TIME_RATIO_LIMIT}×")
run_test("[BFS] Time grows linearly with N (O(N) — not O(N²))", all_bfs_time_ok,
         f"One or more doubling ratios exceeded {TIME_RATIO_LIMIT}×")


--- Time Complexity: O(N) verification ---
  Size          N (cells)     REC (ms)     BFS (ms)
  ------------ ---------- ------------ ------------
  5×5                 25       0.0244       0.0460
  10×10               100       0.1875       0.1802
  20×20               400       0.7141       0.5036
  40×40              1600       1.4379       1.1437

  Doubling pair           REC ratio  BFS ratio   REC O(N)?   BFS O(N)?
  ---------------------- ---------- ---------- ----------- -----------
  5×5 → 10×10                  7.67       3.92      PASS ✓      PASS ✓
  10×10 → 20×20                3.81       2.79      PASS ✓      PASS ✓
  20×20 → 40×40                2.01       2.27      PASS ✓      PASS ✓

  PASS  [REC] Time grows linearly with N (O(N) — not O(N²))
  PASS  [BFS] Time grows linearly with N (O(N) — not O(N²))


In [17]:
# ── SPACE COMPLEXITY — RECURSIVE: call-stack depth grows as O(N) ──────────────
#
# Theory: recursive flood fill uses O(N) stack frames in the worst case
# because it can visit every cell before returning.
#
# What we measure: the actual maximum call-stack depth recorded by
# flood_fill_instrumented() as N increases.
#
# What we assert:
#   1. Max depth > 0 for non-trivial grids (fill actually ran).
#   2. Max depth increases as N increases (not constant or decreasing).
#   3. Max depth is bounded by N — it never exceeds the total cell count.

print("\n--- Space Complexity (Recursive): call-stack depth vs N ---")
print(f"  {'Size':<10} {'N (cells)':>10} {'Max depth':>12} {'≤ N?':>8} {'Depth > 0?':>12}")
print(f"  {'-'*10} {'-'*10} {'-'*12} {'-'*8} {'-'*12}")

rec_space_sizes  = [3, 5, 8, 12, 15]
rec_depths       = []
all_rec_space_ok = True

for n in rec_space_sizes:
    g = create_grid(n, n, default_value=EMPTY)   # all EMPTY — worst case
    n_cells = n * n

    # flood_fill_instrumented returns the max_depth list;
    # [0] extracts the integer from the single-element list
    depth = flood_fill_instrumented(g, 0, 0, EMPTY, 9)[0]
    rec_depths.append(depth)

    # Assertions:
    depth_positive = depth > 0
    depth_bounded  = depth <= n_cells   # must never exceed total cells
    both_ok        = depth_positive and depth_bounded
    if not both_ok: all_rec_space_ok = False

    print(f"  {n}×{n:<7} {n_cells:>10} {depth:>12} "
          f"{'✓' if depth_bounded else '✗':>8} "
          f"{'✓' if depth_positive else '✗':>12}")

# Check that depth grows as N grows (linear trend)
# We verify each depth is larger than the previous one
depth_increases = all(
    rec_depths[i+1] > rec_depths[i]
    for i in range(len(rec_depths) - 1)
)

print()
run_test("[REC] Stack depth > 0 and ≤ N for all grid sizes",
         all_rec_space_ok,
         "Depth was 0 (fill didn't run) or exceeded total cell count")
run_test("[REC] Stack depth increases as N increases (O(N) space growth)",
         depth_increases,
         "Depth did not grow monotonically — space is not scaling linearly")


--- Space Complexity (Recursive): call-stack depth vs N ---
  Size        N (cells)    Max depth     ≤ N?   Depth > 0?
  ---------- ---------- ------------ -------- ------------
  3×3                9            8        ✓            ✓
  5×5               25           24        ✓            ✓
  8×8               64           63        ✓            ✓
  12×12             144          143        ✓            ✓
  15×15             225          224        ✓            ✓

  PASS  [REC] Stack depth > 0 and ≤ N for all grid sizes
  PASS  [REC] Stack depth increases as N increases (O(N) space growth)


In [18]:
# ── SPACE COMPLEXITY — BFS: peak queue size grows as O(N) ────────────────────
#
# Theory: BFS flood fill uses O(N) extra memory for the queue because
# at most every cell can be in the queue at once.
#
# What we measure: the maximum number of cells in the deque at any one
# moment during the BFS, as recorded by flood_fill_queue_instrumented().
#
# What we assert:
#   1. Peak queue size > 0 for non-trivial grids.
#   2. Peak queue size ≤ N — the queue can never hold more cells
#      than exist in the grid.
#   3. Peak queue size increases as N increases.
#   4. Peak queue size is always ≤ recursive max stack depth for the
#      same grid — BFS does not use more memory than recursive.

print("\n--- Space Complexity (BFS): peak queue size vs N ---")
print(f"  {'Size':<10} {'N (cells)':>10} {'Peak queue':>12} {'≤ N?':>8} {'Queue > 0?':>12}")
print(f"  {'-'*10} {'-'*10} {'-'*12} {'-'*8} {'-'*12}")

bfs_space_sizes  = [3, 5, 8, 12, 15]
bfs_peaks        = []
all_bfs_space_ok = True

for n in bfs_space_sizes:
    g = create_grid(n, n, default_value=EMPTY)
    n_cells = n * n

    peak = flood_fill_queue_instrumented(g, 0, 0, EMPTY, 9)
    bfs_peaks.append(peak)

    peak_positive = peak > 0
    peak_bounded  = peak <= n_cells
    both_ok       = peak_positive and peak_bounded
    if not both_ok: all_bfs_space_ok = False

    print(f"  {n}×{n:<7} {n_cells:>10} {peak:>12} "
          f"{'✓' if peak_bounded else '✗':>8} "
          f"{'✓' if peak_positive else '✗':>12}")

# Check that peak queue size grows as N grows
bfs_peaks_increase = all(
    bfs_peaks[i+1] > bfs_peaks[i]
    for i in range(len(bfs_peaks) - 1)
)

print()
run_test("[BFS] Peak queue size > 0 and ≤ N for all grid sizes",
         all_bfs_space_ok,
         "Queue was empty or exceeded total cell count")
run_test("[BFS] Peak queue size increases as N increases (O(N) space growth)",
         bfs_peaks_increase,
         "Queue peak did not grow monotonically — space is not scaling linearly")


--- Space Complexity (BFS): peak queue size vs N ---
  Size        N (cells)   Peak queue     ≤ N?   Queue > 0?
  ---------- ---------- ------------ -------- ------------
  3×3                9            3        ✓            ✓
  5×5               25            5        ✓            ✓
  8×8               64            8        ✓            ✓
  12×12             144           12        ✓            ✓
  15×15             225           15        ✓            ✓

  PASS  [BFS] Peak queue size > 0 and ≤ N for all grid sizes
  PASS  [BFS] Peak queue size increases as N increases (O(N) space growth)


In [19]:
# ── SPACE COMPARISON: BFS queue vs Recursive stack ────────────────────────────
#
# Both algorithms are O(N) space, but the CONSTANT matters in practice.
#
# Recursive: each stack frame stores the function's local variables
#   (grid reference, row, col, target_color, fill_color, depth, max_depth)
#   plus Python's own frame overhead (~hundreds of bytes per frame).
#
# BFS: the queue stores only (row, col) tuples — 2 integers per cell.
#   Much smaller per entry than a full stack frame.
#
# We use tracemalloc to measure actual peak memory allocated by each
# algorithm on the same grid, then assert BFS uses ≤ recursive memory.

print("\n--- Space Comparison: actual peak memory (tracemalloc) ---")
print(f"  {'Size':<10} {'N':>6} {'REC peak (KB)':>14} {'BFS peak (KB)':>14} {'BFS ≤ REC?':>12}")
print(f"  {'-'*10} {'-'*6} {'-'*14} {'-'*14} {'-'*12}")

memory_sizes     = [5, 8, 10, 12]
all_memory_ok    = True

for n in memory_sizes:
    n_cells = n * n

    # ── Measure recursive peak memory ──
    g_rec = create_grid(n, n, default_value=EMPTY)
    tracemalloc.start()                        # begin tracking allocations
    flood_fill_instrumented(g_rec, 0, 0, EMPTY, 9)
    _, rec_peak = tracemalloc.get_traced_memory()  # (current, peak) in bytes
    tracemalloc.stop()                         # stop tracking

    # ── Measure BFS peak memory ──
    g_bfs = create_grid(n, n, default_value=EMPTY)
    tracemalloc.start()
    flood_fill_queue_instrumented(g_bfs, 0, 0, EMPTY, 9)
    _, bfs_peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    # Convert bytes → kilobytes for readability
    rec_kb = rec_peak / 1024
    bfs_kb = bfs_peak / 1024

    # BFS should use less or equal memory than recursive
    mem_ok = bfs_peak <= rec_peak
    if not mem_ok: all_memory_ok = False

    print(f"  {n}×{n:<7} {n_cells:>6} {rec_kb:>14.2f} {bfs_kb:>14.2f} "
          f"{'PASS ✓' if mem_ok else 'FAIL ✗':>12}")

print()
run_test("[SPACE] BFS peak memory ≤ Recursive peak memory on same grid",
         all_memory_ok,
         "BFS used more memory than recursive — unexpected result")

# Restore the original recursion limit now that all complexity tests are done
sys.setrecursionlimit(_old_limit)


--- Space Comparison: actual peak memory (tracemalloc) ---
  Size            N  REC peak (KB)  BFS peak (KB)   BFS ≤ REC?
  ---------- ------ -------------- -------------- ------------
  5×5           25           0.01           0.82       FAIL ✗
  8×8           64           0.01           1.34       FAIL ✗
  10×10         100           0.01           1.34       FAIL ✗
  12×12         144           0.01           1.34       FAIL ✗

  FAIL  [SPACE] BFS peak memory ≤ Recursive peak memory on same grid → BFS used more memory than recursive — unexpected result


---
## 8. Final summary

In [20]:
summary()
summary()


════════════════════════════════════════════════════════════
  RESULTS   41 passed  |  3 failed  |  44 total
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  RESULTS   41 passed  |  3 failed  |  44 total
════════════════════════════════════════════════════════════


# Display the fill spreading step-by-step + allow user choose the fill start point 

In [21]:
# Create a fresh demo grid 
user_test_grid = create_grid(6, 7)
set_cell(user_test_grid, 0, 1, WALL)
set_cell(user_test_grid, 1, 2, WALL)
set_cell(user_test_grid, 2, 2, FILLED)
print_grid(user_test_grid)
print(len(user_test_grid[0]))

0 1 0 0 0 0 0
0 0 1 0 0 0 0
0 0 2 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
7


## Allow user choose the start point and end point to fill 


### Input helper

**`prompt_index(label, max_value)`**

- `label`: tells the prompt whether the user is entering a row or column.
- `max_value`: highest valid 1-based index for that dimension.
- The function converts the user's 1-based input into Python's 0-based index.

### Start point selection

**`choose_fill_start_point(grid)`**

- Prints the grid, dimensions, and value legend before asking for input.
- Stores the chosen position as `s_row` and `s_col`.
- Reads `cell_value` to reject invalid cells and wall cells.
- Returns the first valid non-wall start cell.
- Saves the result into `user_start_row` and `user_start_col`.

### End point selection

**`choose_fill_end_point(grid, start_row, start_col)`**

- Prints the same grid context before asking for the ending cell.
- Stores the chosen position as `e_row` and `e_col`.
- Reads `cell_value` to reject invalid cells and wall cells.
- Rejects an end point that is the same as the start point.
- Returns the first valid non-wall end cell.
- Saves the result into `user_end_row` and `user_end_col`.


In [22]:
"""
    Ask the user for one index value (row or column).
    Return the value as a zero-based index for Python list access.
"""
def prompt_index(label, max_value):
    while True:

        value = input(f"Enter the {label} index (from 1 to {max_value}): ").strip()
        if value == "":
            print(f"{label.capitalize()} index cannot be empty. Please enter a valid number.")
        elif not value.isdigit():
            print(f"Invalid input. Please enter a valid number for the {label} index.")
        elif not (1 <= int(value) <= max_value):
            print(f"{label.capitalize()} index is out of bounds. Please enter a valid {label} index.")
        else:

            return int(value) - 1

    """
    Let the user choose a valid starting cell for flood fill.
    The chosen cell must be inside the grid and must not be a WALL.
    """
def choose_fill_start_point(grid):


    print_grid(grid)
    print("Grid dimensions:", len(grid), "x", len(grid[0]))
    print("0=EMPTY, 1=WALL, 2=FILLED")

    while True:

        start_point = ()
        print("--- START POINT ---")
        s_row = prompt_index("row", len(grid))
        s_col = prompt_index("column", len(grid[0]))

        cell_value = get_cell(grid, s_row, s_col)

        if cell_value is None:
            print("Invalid cell position. Please enter valid row and column indices.")
        elif cell_value == WALL:
            print("Cannot fill a WALL cell. Please choose a different cell.")
        else:

            print(f"Entered start row number: {s_row+1} \nEntered start column number: {s_col+1}")
            return s_row, s_col

user_start_row, user_start_col = choose_fill_start_point(user_test_grid)

def choose_fill_end_point(grid, start_row, start_col):


    print_grid(grid)
    print("Grid dimensions:", len(grid), "x", len(grid[0]))
    print("0=EMPTY, 1=WALL, 2=FILLED")

    while True:

        end_point = ()
        print("--- END POINT ---")
        e_row = prompt_index("row", len(grid))
        e_col = prompt_index("column", len(grid[0]))

        cell_value = get_cell(grid, e_row, e_col)

        if cell_value is None:
            print("Invalid cell position. Please enter valid row and column indices.")
        elif cell_value == WALL:
            print("Cannot fill a WALL cell. Please choose a different cell.")
        elif start_row == e_row and start_col == e_col:
            print("End point cannot be the same as start point. Please choose a different cell.")
        else:

            print(f"Enter end row number: {e_row+1} \nEntered end column number: {e_col+1}")
            return e_row, e_col

user_end_row, user_end_col = choose_fill_end_point(user_test_grid, user_start_row, user_start_col)


0 1 0 0 0 0 0
0 0 1 0 0 0 0
0 0 2 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
Grid dimensions: 6 x 7
0=EMPTY, 1=WALL, 2=FILLED
--- START POINT ---
Entered start row number: 1 
Entered start column number: 1
0 1 0 0 0 0 0
0 0 1 0 0 0 0
0 0 2 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
Grid dimensions: 6 x 7
0=EMPTY, 1=WALL, 2=FILLED
--- END POINT ---
Enter end row number: 1 
Entered end column number: 3


## Testing with user input 


In [23]:
flood_fill_queue(user_test_grid, user_start_row, user_start_col, 0, 4)
print_grid(user_test_grid)

4 1 4 4 4 4 4
4 4 1 4 4 4 4
4 4 2 4 4 4 4
4 4 4 4 4 4 4
4 4 4 4 4 4 4
4 4 4 4 4 4 4


## DISPLAYING THE FILL STEP-BY-STEP

In [24]:
import time 

### Validation

- Checks that the start and end positions are inside the grid.
- Stops early if either the start cell or end cell is a wall.

### BFS data structures

- `queue`: frontier of cells that still need neighbor checks.
- `visited`: cells that have already been explored.
- `parent`: maps each discovered cell to the cell that discovered it, so the path can be rebuilt later.

### Initialization

- Adds the start cell to `queue` and `visited`.
- Stores `None` as the parent of the start cell.
- Marks the start cell with `visited_color`.
- Prints the first frame so the search is visible immediately.

### Search loop

- Repeatedly pops the next frontier cell from the queue.
- Stops when the current cell is the end cell.
- Checks four orthogonal neighbors: up, down, left, and right.
- For each valid unvisited non-wall neighbor:
  - marks it as visited,
  - stores its parent,
  - adds it to the queue,
  - paints it with `visited_color`,
  - prints the next animation frame.

### Path reconstruction

- If the end cell is found, traces backward through `parent` from end to start.
- Reverses that list so animation runs from start to end.
- Repaints each path cell with `path_color` step by step.
- If BFS finishes without reaching the end cell, prints that no path was found.


In [25]:
def flood_fill_queue_steps(grid, start_row, start_col, end_row, end_col, visited_color, path_color, delay=0.5):

    if not is_valid_cell(grid, start_row, start_col) or not is_valid_cell(grid, end_row, end_col) or grid[start_row][start_col] == WALL or grid[end_row][end_col] == WALL:
        return




    queue = deque()
    visited = set()
    parent = {}


    queue.append((start_row, start_col))
    visited.add((start_row,start_col))
    parent[(start_row,start_col)] = None
    grid[start_row][start_col] = visited_color


    print_grid(grid)
    print("\n")
    time.sleep(delay)


    found = False
    while len(queue) > 0:

        current_row, current_col = queue.popleft()
        if (current_row, current_col) == (end_row, end_col):
            found = True
            break


        neighbors = [
            (current_row - 1, current_col),
            (current_row + 1, current_col),
            (current_row, current_col - 1),
            (current_row, current_col + 1),
        ]

        for r, c in neighbors:
            if is_valid_cell(grid, r, c) and grid[r][c] != WALL and (r, c) not in visited:
                visited.add((r,c))
                parent[(r,c)] = (current_row, current_col)
                queue.append((r, c))
                grid[r][c] = visited_color


                print_grid(grid)
                print()
                time.sleep(delay)



    if found == True:
        current_trace = (end_row, end_col)
        path_reconstruction = []

        while current_trace != None:
            path_reconstruction.append(current_trace)
            current_trace = parent[current_trace]

        path_reconstruction.reverse()

        print("Now it's time to visualize the path!")
        print()

        for row, col in path_reconstruction:
            grid[row][col] = path_color
            print_grid(grid)
            print()
            time.sleep(delay)

        print("Done!")
    else:
        print("Cannot find the path!")


### Step-by-step demo grid

Creates a fresh `6 x 7` grid, adds two wall cells, and prints the grid before running the BFS animation.


In [26]:
new_test_grid = create_grid(6, 7)
set_cell(new_test_grid, 0, 1, WALL)
set_cell(new_test_grid, 1, 2, WALL)
print_grid(new_test_grid)


0 1 0 0 0 0 0
0 0 1 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0


In [27]:
flood_fill_queue_steps(new_test_grid, 0, 0, 2, 4, 2, "X", delay=0.5)

2 1 0 0 0 0 0
0 0 1 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0


2 1 0 0 0 0 0
2 0 1 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0

2 1 0 0 0 0 0
2 0 1 0 0 0 0
2 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0

2 1 0 0 0 0 0
2 2 1 0 0 0 0
2 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0

2 1 0 0 0 0 0
2 2 1 0 0 0 0
2 0 0 0 0 0 0
2 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0

2 1 0 0 0 0 0
2 2 1 0 0 0 0
2 2 0 0 0 0 0
2 0 0 0 0 0 0
0 0 0 0 0 0 0
0 0 0 0 0 0 0

2 1 0 0 0 0 0
2 2 1 0 0 0 0
2 2 0 0 0 0 0
2 0 0 0 0 0 0
2 0 0 0 0 0 0
0 0 0 0 0 0 0

2 1 0 0 0 0 0
2 2 1 0 0 0 0
2 2 0 0 0 0 0
2 2 0 0 0 0 0
2 0 0 0 0 0 0
0 0 0 0 0 0 0

2 1 0 0 0 0 0
2 2 1 0 0 0 0
2 2 2 0 0 0 0
2 2 0 0 0 0 0
2 0 0 0 0 0 0
0 0 0 0 0 0 0

2 1 0 0 0 0 0
2 2 1 0 0 0 0
2 2 2 0 0 0 0
2 2 0 0 0 0 0
2 0 0 0 0 0 0
2 0 0 0 0 0 0

2 1 0 0 0 0 0
2 2 1 0 0 0 0
2 2 2 0 0 0 0
2 2 0 0 0 0 0
2 2 0 0 0 0 0
2 0 0 0 0 0 0

2 1 0 0 0 0 0
2 2 1 0 0 0 0
2 2 2 0 0 0 0
2 2 2 0 0 0 0
2 2 0 0 

# PYGAME SET UP

In [28]:
!pip install pygame

In [29]:
import pygame

CELL_SIZE = 80      # Width and height of each grid cell in pixels
MARGIN    = 1       # Gap between cells (acts as grid lines)
FPS       = 50      # Target frames per second
# Cell value → RGB color mapping 
COLOR_MAP = {
    0: (255, 255, 255),  # EMPTY   - Light gray
    1: (0, 0, 0),  # WALL    - Black
    2: (0, 255, 0),  # FILLED  - Lime
    3: (220, 0, 0),  # RED
    9: (255, 255, 0),  # YELLOW
}
COLOR_FALLBACK  = (128, 128, 128)  # Used when a cell value has no entry in COLOR_MAP
COLOR_GRID_BG   = (105, 105, 105)  # Window background color (visible as grid lines)
# ==============================================================
# GRID GEOMETRY HELPERS
# ==============================================================
def get_window_size(grid):
    """Return the (width, height) in pixels required to display the grid."""
    rows = len(grid)
    cols = len(grid[0])
    width  = cols * CELL_SIZE + (cols + 1) * MARGIN
    height = rows * CELL_SIZE + (rows + 1) * MARGIN
    return width, height
def cell_rect(row, col):
    """
    Return a pygame.Rect for the cell at (row, col).
    """
    x = MARGIN + col * (CELL_SIZE + MARGIN)
    y = MARGIN + row * (CELL_SIZE + MARGIN)
    return pygame.Rect(x, y, CELL_SIZE, CELL_SIZE)
                       
# ==============================================================
# MOUSE CLICK TO GRID CELL MAPPING
# This helper function converts the mouse click position in pixels
# into the correct row and column index in the grid.
# ==============================================================

def mouse_to_grid_cell(mouse_pos, grid):
    """
    Convert mouse pixel position to grid row and column.

    Parameters:
    mouse_pos: the mouse position returned by Pygame, for example (x, y)
    grid: the current 2D grid

    Returns:
    (row, col) if the click is inside the grid
    None if the click is outside the grid
    """

    # Separate the mouse position into x and y coordinates
    mouse_x, mouse_y = mouse_pos

    # Each grid cell takes CELL_SIZE pixels plus MARGIN pixels
    cell_block_size = CELL_SIZE + MARGIN

    # Convert x position to column index
    col = mouse_x // cell_block_size

    # Convert y position to row index
    row = mouse_y // cell_block_size

    # Check if the calculated row and column are inside the grid
    if is_valid_cell(grid, row, col):
        return int(row), int(col)

    # Return None if the click is outside the valid grid area
    return None
# ==============================================================
# RENDERING
# ==============================================================
def draw_grid(screen, grid):
    """
    Draw every cell in the grid onto the given surface.
    Cell color is determined by its value via COLOR_MAP.
    """
    for r, row in enumerate(grid):
        for c, value in enumerate(row):
            color = COLOR_MAP.get(value, COLOR_FALLBACK)
            pygame.draw.rect(screen, color, cell_rect(r, c))
# ==============================================================
# PYGAME INITIALIZATION
# ==============================================================
def init_pygame(grid, title="Flood Fill Visualizer"):
    """
    Initialize pygame and create a window sized to fit the grid exactly.
    Returns (screen, clock) for use in the main loop.
    """
    pygame.init()
    width, height = get_window_size(grid)
    screen = pygame.display.set_mode((width, height))
    pygame.display.set_caption("Floodfill Research Pygame Visualization")
    clock = pygame.time.Clock()
    return screen, clock
# ==============================================================
# MAIN LOOP
# ==============================================================
def run_display(grid):
    """
    Run the pygame main loop and render the grid each frame.
    Exit: close the window or press ESC.
    """
    screen, clock = init_pygame(grid)
    running = True
    while running:
        # -- Event handling --
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                running = False
# ==================================================
# DETECT LEFT MOUSE CLICK
# This block checks if the user clicks the left mouse button.
# If the click is valid, it maps the pixel position to a grid cell.
# ==================================================

            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                # Get the clicked grid cell from the mouse pixel position
                clicked_cell = mouse_to_grid_cell(event.pos, grid)

                # Check if the click is inside a valid grid cell
                if clicked_cell is not None:
                    # Separate the result into row and column
                    row, col = clicked_cell

                    # Print the original mouse pixel position
                    print("Clicked pixel:", event.pos)

                    # Print the mapped grid cell position
                    print("Mapped to grid cell:", row, col)

                    # Print the value stored in that grid cell
                    print("Cell value:", grid[row][col])
        # -- Render --
        screen.fill(COLOR_GRID_BG)
        draw_grid(screen, grid)
        pygame.display.flip()
        clock.tick(FPS)
    pygame.quit()


pygame 2.6.1 (SDL 2.28.4, Python 3.13.1)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [30]:
# QUICK TEST  

test_grid = create_grid(6, 6)
set_cell(test_grid, 0, 2, WALL)
set_cell(test_grid, 1, 2, WALL)
set_cell(test_grid, 2, 2, WALL)
set_cell(test_grid, 2, 3, FILLED)
set_cell(test_grid, 3, 1, WALL)
run_display(test_grid)

KeyboardInterrupt: 

In [ ]:
# ============================================================
# PYGAME BFS PATHFINDING VISUALIZER
# ============================================================

import pygame
import sys
from collections import deque
import copy

# ============================================================
# BASIC GRID CONSTANTS
# ============================================================

EMPTY   = 0
WALL    = 1
FILLED  = 2
START   = 3
END     = 4
VISITED = 5
PATH    = 6

ROWS = 12
COLS = 16

CELL_SIZE = 45
MARGIN = 2

FPS = 60
BFS_DELAY = 60

# ============================================================
# COLORS
# ============================================================

COLOR_MAP = {
    EMPTY   : (255,255,255),
    WALL    : (0,0,0),
    FILLED  : (0,255,0),
    START   : (0,200,100),
    END     : (220,0,0),
    VISITED : (65,105,225),
    PATH    : (255,215,0),
}

COLOR_FALLBACK = (128,128,128)
COLOR_GRID_BG = (70,70,70)

PANEL_COLOR = (30,30,40)
TEXT_COLOR = (240,240,240)

# ============================================================
# WINDOW SIZE
# ============================================================

PANEL_WIDTH = 220

GRID_WIDTH = COLS * CELL_SIZE + (COLS + 1) * MARGIN
GRID_HEIGHT = ROWS * CELL_SIZE + (ROWS + 1) * MARGIN

WINDOW_WIDTH = GRID_WIDTH + PANEL_WIDTH
WINDOW_HEIGHT = GRID_HEIGHT

# ============================================================
# MODES
# ============================================================

MODE_WALL = "wall"
MODE_ERASE = "erase"
MODE_START = "start"
MODE_END = "end"

# ============================================================
# GRID FUNCTIONS
# ============================================================

def create_grid(rows, cols):
    return [[EMPTY for _ in range(cols)] for _ in range(rows)]

def get_cell(grid, row, col):
    if 0 <= row < len(grid) and 0 <= col < len(grid[0]):
        return grid[row][col]
    return None

def set_cell(grid, row, col, value):
    if 0 <= row < len(grid) and 0 <= col < len(grid[0]):
        grid[row][col] = value

def is_valid_cell(grid, row, col):
    return 0 <= row < len(grid) and 0 <= col < len(grid[0])

# ============================================================
# GEOMETRY HELPERS
# ============================================================

def cell_rect(row, col):

    x = MARGIN + col * (CELL_SIZE + MARGIN)
    y = MARGIN + row * (CELL_SIZE + MARGIN)

    return pygame.Rect(x, y, CELL_SIZE, CELL_SIZE)

def mouse_to_grid_cell(mouse_pos, grid):

    mouse_x, mouse_y = mouse_pos

    if mouse_x >= GRID_WIDTH or mouse_y >= GRID_HEIGHT:
        return None

    block_size = CELL_SIZE + MARGIN

    col = mouse_x // block_size
    row = mouse_y // block_size

    if is_valid_cell(grid, row, col):
        return int(row), int(col)

    return None

# ============================================================
# DRAW FUNCTIONS
# ============================================================

def draw_grid(screen, grid):

    for r,row in enumerate(grid):
        for c,value in enumerate(row):

            color = COLOR_MAP.get(value, COLOR_FALLBACK)

            pygame.draw.rect(
                screen,
                color,
                cell_rect(r,c),
                border_radius=4
            )

def draw_panel(screen, font, mode):

    pygame.draw.rect(
        screen,
        PANEL_COLOR,
        (GRID_WIDTH,0,PANEL_WIDTH,WINDOW_HEIGHT)
    )

    title = font.render(
        "BFS VISUALIZER",
        True,
        TEXT_COLOR
    )

    screen.blit(title,(GRID_WIDTH + 20,20))

    mode_text = font.render(
        f"MODE : {mode.upper()}",
        True,
        TEXT_COLOR
    )

    screen.blit(mode_text,(GRID_WIDTH + 20,60))

    # ========================================================
    # CONTROLS
    # ========================================================

    controls = [

        "CONTROLS",
        "W = Draw Wall",
        "E = Erase",
        "S = Set Start",
        "D = Set End",
        "SPACE = Run BFS",
        "R = Reset",
        "ESC = Quit"

    ]

    y = 120

    for line in controls:

        text = font.render(line,True,TEXT_COLOR)

        screen.blit(text,(GRID_WIDTH + 20,y))

        y += 28

    # ========================================================
    # COLOR LEGEND
    # ========================================================

    legend_title = font.render(
        "COLOR LEGEND",
        True,
        TEXT_COLOR
    )

    screen.blit(legend_title,(GRID_WIDTH + 20,360))

    items = [

        (COLOR_MAP[START],   "Start cell"),
        (COLOR_MAP[END],     "End cell"),
        (COLOR_MAP[WALL],    "Wall"),
        (COLOR_MAP[FILLED],  "Flood-filled"),
        (COLOR_MAP[VISITED], "Visited (BFS)"),
        (COLOR_MAP[PATH],    "Shortest path"),

    ]

    y = 400

    for color,label in items:

        pygame.draw.rect(
            screen,
            color,
            (GRID_WIDTH + 20,y,20,20),
            border_radius=4
        )

        text = font.render(
            label,
            True,
            TEXT_COLOR
        )

        screen.blit(text,(GRID_WIDTH + 50,y))

        y += 35

# ============================================================
# BFS GENERATOR
# ============================================================

def bfs_path_generator(grid, start, end):

    sr, sc = start
    er, ec = end

    queue = deque()
    visited = set()
    parent = {}

    queue.append((sr,sc))

    visited.add((sr,sc))

    parent[(sr,sc)] = None

    while len(queue) > 0:

        current = queue.popleft()

        current_row, current_col = current

        if current == (er,ec):

            path = []

            cur = current

            while cur is not None:

                path.append(cur)

                cur = parent[cur]

            path.reverse()

            for cell in path:

                yield ("path",cell[0],cell[1])

            return

        neighbors = [

            (current_row - 1,current_col),
            (current_row + 1,current_col),
            (current_row,current_col - 1),
            (current_row,current_col + 1)

        ]

        for r,c in neighbors:

            if not is_valid_cell(grid,r,c):
                continue

            value = get_cell(grid,r,c)

            if value in (WALL,VISITED):
                continue

            if (r,c) in visited:
                continue

            visited.add((r,c))

            parent[(r,c)] = (current_row,current_col)

            queue.append((r,c))

            yield ("visit",r,c)

    yield ("no_path",)

# ============================================================
# MAIN PROGRAM
# ============================================================

def run_visualizer():

    pygame.init()

    screen = pygame.display.set_mode(
        (WINDOW_WIDTH,WINDOW_HEIGHT)
    )

    pygame.display.set_caption(
        "Flood Fill BFS Pathfinder"
    )

    clock = pygame.time.Clock()

    font = pygame.font.SysFont("consolas",18)

    grid = create_grid(ROWS,COLS)

    original_grid = copy.deepcopy(grid)

    mode = MODE_WALL

    start_cell = None
    end_cell = None

    bfs_running = False
    bfs_generator = None

    bfs_timer = 0

    running = True

    while running:

        dt = clock.tick(FPS)

        # ====================================================
        # EVENTS
        # ====================================================

        for event in pygame.event.get():

            if event.type == pygame.QUIT:
                running = False

            if event.type == pygame.KEYDOWN:

                if event.key == pygame.K_ESCAPE:
                    running = False

                if event.key == pygame.K_w:
                    mode = MODE_WALL

                if event.key == pygame.K_e:
                    mode = MODE_ERASE

                if event.key == pygame.K_s:
                    mode = MODE_START

                if event.key == pygame.K_d:
                    mode = MODE_END

                if event.key == pygame.K_r:

                    grid = copy.deepcopy(original_grid)

                    start_cell = None
                    end_cell = None

                    bfs_running = False

                if event.key == pygame.K_SPACE:

                    if start_cell and end_cell:

                        for r in range(ROWS):
                            for c in range(COLS):

                                if get_cell(grid,r,c) in (VISITED,PATH):

                                    set_cell(grid,r,c,EMPTY)

                        bfs_generator = bfs_path_generator(
                            grid,
                            start_cell,
                            end_cell
                        )

                        bfs_running = True

            # ====================================================
            # MOUSE DRAWING
            # ====================================================

            if event.type == pygame.MOUSEBUTTONDOWN:

                if event.button == 1 and not bfs_running:

                    clicked = mouse_to_grid_cell(
                        event.pos,
                        grid
                    )

                    if clicked:

                        row,col = clicked

                        # WALL MODE
                        if mode == MODE_WALL:

                            if get_cell(grid,row,col) not in (START,END):

                                set_cell(grid,row,col,WALL)

                        # ERASE MODE
                        elif mode == MODE_ERASE:

                            if start_cell == (row,col):
                                start_cell = None

                            if end_cell == (row,col):
                                end_cell = None

                            set_cell(grid,row,col,EMPTY)

                        # START MODE
                        elif mode == MODE_START:

                            if start_cell:

                                set_cell(
                                    grid,
                                    start_cell[0],
                                    start_cell[1],
                                    EMPTY
                                )

                            start_cell = (row,col)

                            set_cell(grid,row,col,START)

                        # END MODE
                        elif mode == MODE_END:

                            if end_cell:

                                set_cell(
                                    grid,
                                    end_cell[0],
                                    end_cell[1],
                                    EMPTY
                                )

                            end_cell = (row,col)

                            set_cell(grid,row,col,END)

        # ====================================================
        # BFS ANIMATION
        # ====================================================

        if bfs_running and bfs_generator is not None:

            bfs_timer += dt

            if bfs_timer >= BFS_DELAY:

                bfs_timer = 0

                try:

                    step = next(bfs_generator)

                    if step[0] == "visit":

                        _,r,c = step

                        if get_cell(grid,r,c) == EMPTY:

                            set_cell(grid,r,c,VISITED)

                    elif step[0] == "path":

                        _,r,c = step

                        if get_cell(grid,r,c) not in (START,END):

                            set_cell(grid,r,c,PATH)

                    elif step[0] == "no_path":

                        print("No path found")

                        bfs_running = False

                except StopIteration:

                    bfs_running = False

        # ====================================================
        # RENDER
        # ====================================================

        screen.fill(COLOR_GRID_BG)

        draw_grid(screen,grid)

        draw_panel(screen,font,mode)

        pygame.display.flip()

    pygame.quit()
    sys.exit()

# ============================================================
# START PROGRAM
# ============================================================

run_visualizer()

No path found
No path found


SystemExit: 

---
## Color Picker

Standalone demo: bottom picker bar, highlighted editing state, and popup palette after choosing a target.


### Demo overview

Standalone Pygame demo with a grid, side controls, a bottom color picker bar, and a popup palette for changing target colors.

### Constants and state

- **Basic cell constants:** numeric values for empty cells, walls, fill cells, start/end cells, visited cells, and path cells.
- **Grid/window constants:** row count, column count, cell size, margins, FPS, animation delay, panel size, and picker bar size.
- **Color maps:** default colors for each cell type, UI colors, fallback color, and selectable palette colors.
- **Modes:** wall paint, fill, erase, start selection, and end selection.

### Helper sections

- **Grid helpers:** create grids, read/write cells, validate coordinates, and map mouse clicks to grid cells.
- **Color picker layout:** builds rectangles for target buttons, swatches, the editing indicator, and the popup palette.
- **Drawing:** renders the grid, side panel, controls, picker bar, active color target, and palette choices.

### Pathfinding and main loop

- **BFS pathfinding:** finds start/end cells, explores the grid, tracks parents, and yields animation states.
- **Main demo:** initializes Pygame, handles keyboard/mouse input, updates selected colors and modes, animates BFS, and redraws the UI every frame.


In [3]:
import pygame
from collections import deque





EMPTY   = 0
WALL    = 1
FILLED  = 2
START   = 3
END     = 4
VISITED = 5
PATH    = 6

ROWS = 12
COLS = 16
CELL_SIZE = 45
MARGIN = 2
FPS = 60
BFS_DELAY = 45

PANEL_WIDTH = 220
BOTTOM_PICKER_HEIGHT = 140

GRID_WIDTH = COLS * CELL_SIZE + (COLS + 1) * MARGIN
GRID_HEIGHT = ROWS * CELL_SIZE + (ROWS + 1) * MARGIN
WINDOW_WIDTH = GRID_WIDTH + PANEL_WIDTH
WINDOW_HEIGHT = GRID_HEIGHT + BOTTOM_PICKER_HEIGHT





DEFAULT_COLOR_MAP = {
    EMPTY   : (255, 255, 255),
    WALL    : (20, 20, 24),
    FILLED  : (32, 201, 151),
    START   : (46, 204, 113),
    END     : (231, 76, 60),
    VISITED : (65, 105, 225),
    PATH    : (255, 209, 102),
}

COLOR_MAP = DEFAULT_COLOR_MAP.copy()

COLOR_FALLBACK = (128, 128, 128)
COLOR_GRID_BG = (78, 82, 92)
PANEL_COLOR = (32, 35, 44)
BOTTOM_BAR_COLOR = (26, 29, 37)
BUTTON_COLOR = (48, 53, 66)
BUTTON_ACTIVE = (76, 91, 112)
BORDER_COLOR = (110, 118, 138)
TEXT_COLOR = (238, 241, 245)
MUTED_TEXT = (170, 177, 190)
HIGHLIGHT = (255, 255, 255)
ALERT_COLOR = (255, 185, 85)
ALERT_BG = (82, 56, 35)

PALETTE = [
    ("Red",    (231, 76, 60)),
    ("Orange", (243, 156, 18)),
    ("Yellow", (255, 209, 102)),
    ("Green",  (46, 204, 113)),
    ("Mint",   (32, 201, 151)),
    ("Cyan",   (0, 188, 212)),
    ("Blue",   (65, 105, 225)),
    ("Purple", (155, 89, 182)),
    ("Pink",   (236, 72, 153)),
    ("Gray",   (149, 165, 166)),
    ("Black",  (20, 20, 24)),
    ("White",  (255, 255, 255)),
]

COLOR_TARGETS = [
    (WALL, "Wall"),
    (FILLED, "Fill"),
    (START, "Start"),
    (END, "End"),
    (VISITED, "Visited"),
    (PATH, "Path"),
]





MODE_WALL = "wall"
MODE_FILL = "fill"
MODE_ERASE = "erase"
MODE_START = "start"
MODE_END = "end"

MODE_LABELS = {
    MODE_WALL: "Wall",
    MODE_FILL: "Fill",
    MODE_ERASE: "Erase",
    MODE_START: "Start",
    MODE_END: "End",
}





def create_grid(rows, cols):
    return [[EMPTY for _ in range(cols)] for _ in range(rows)]


def get_cell(grid, row, col):
    if 0 <= row < len(grid) and 0 <= col < len(grid[0]):
        return grid[row][col]
    return None


def set_cell(grid, row, col, value):
    if 0 <= row < len(grid) and 0 <= col < len(grid[0]):
        grid[row][col] = value


def is_valid_cell(grid, row, col):
    return 0 <= row < len(grid) and 0 <= col < len(grid[0])


def mouse_to_grid_cell(pos):
    mouse_x, mouse_y = pos

    if mouse_x < 0 or mouse_x >= GRID_WIDTH:
        return None

    if mouse_y < 0 or mouse_y >= GRID_HEIGHT:
        return None

    col = mouse_x // (CELL_SIZE + MARGIN)
    row = mouse_y // (CELL_SIZE + MARGIN)

    if row >= ROWS or col >= COLS:
        return None

    cell_x = MARGIN + col * (CELL_SIZE + MARGIN)
    cell_y = MARGIN + row * (CELL_SIZE + MARGIN)
    cell_rect = pygame.Rect(cell_x, cell_y, CELL_SIZE, CELL_SIZE)

    if cell_rect.collidepoint(pos):
        return row, col

    return None





def get_color_picker_layout():
    bottom_y = GRID_HEIGHT

    editing_rect = pygame.Rect(20, bottom_y + 72, 170, 48)

    target_rects = []
    target_x = 215
    target_y = bottom_y + 58
    target_w = 86
    target_h = 28
    target_gap = 8

    for index, (target, label) in enumerate(COLOR_TARGETS):
        x = target_x + index * (target_w + target_gap)
        rect = pygame.Rect(x, target_y, target_w, target_h)
        target_rects.append((rect, target, label))

    popup_rect = pygame.Rect(target_x, bottom_y + 92, 476, 40)

    swatch_rects = []
    swatch_x = popup_rect.x + 12
    swatch_y = popup_rect.y + 8
    swatch_size = 24
    swatch_gap = 7

    for index, (name, color) in enumerate(PALETTE):
        x = swatch_x + index * (swatch_size + swatch_gap)
        rect = pygame.Rect(x, swatch_y, swatch_size, swatch_size)
        swatch_rects.append((rect, name, color))

    return target_rects, swatch_rects, editing_rect, popup_rect


def color_choice_is_blocked(target, color):
    path_would_match_wall = target == PATH and color == COLOR_MAP[WALL]
    wall_would_match_path = target == WALL and color == COLOR_MAP[PATH]
    return path_would_match_wall or wall_would_match_path





def draw_grid(screen, grid):
    for row in range(ROWS):
        for col in range(COLS):
            value = grid[row][col]
            color = COLOR_MAP.get(value, COLOR_FALLBACK)

            x = MARGIN + col * (CELL_SIZE + MARGIN)
            y = MARGIN + row * (CELL_SIZE + MARGIN)
            rect = pygame.Rect(x, y, CELL_SIZE, CELL_SIZE)
            pygame.draw.rect(screen, color, rect)


def draw_panel(screen, font, small_font, mode):
    panel_rect = pygame.Rect(GRID_WIDTH, 0, PANEL_WIDTH, GRID_HEIGHT)
    pygame.draw.rect(screen, PANEL_COLOR, panel_rect)

    title = font.render("Flood Fill Demo", True, TEXT_COLOR)
    screen.blit(title, (GRID_WIDTH + 18, 22))

    mode_text = small_font.render("Mode: " + MODE_LABELS[mode], True, TEXT_COLOR)
    screen.blit(mode_text, (GRID_WIDTH + 18, 62))

    controls = [
        "W  Wall paint",
        "F  Fill paint",
        "E  Erase",
        "S  Set start",
        "D  Set end",
        "SPACE  Run BFS",
        "R  Reset grid",
        "C  Reset colors",
        "ESC  Quit",
    ]

    y = 110
    for line in controls:
        text = small_font.render(line, True, MUTED_TEXT)
        screen.blit(text, (GRID_WIDTH + 18, y))
        y += 25

    legend_title = small_font.render("Color Legend", True, TEXT_COLOR)
    screen.blit(legend_title, (GRID_WIDTH + 18, 360))

    legend_items = [
        (WALL, "Wall"),
        (FILLED, "Fill"),
        (START, "Start"),
        (END, "End"),
        (VISITED, "Visited"),
        (PATH, "Path"),
    ]

    y = 395
    for value, label in legend_items:
        color = COLOR_MAP.get(value, COLOR_FALLBACK)
        swatch = pygame.Rect(GRID_WIDTH + 18, y, 22, 22)
        pygame.draw.rect(screen, color, swatch, border_radius=4)
        pygame.draw.rect(screen, BORDER_COLOR, swatch, 1, border_radius=4)

        text = small_font.render(label, True, TEXT_COLOR)
        screen.blit(text, (GRID_WIDTH + 50, y + 2))
        y += 30


def draw_color_picker(screen, font, small_font, selected_target, palette_open, picker_message):
    bottom_rect = pygame.Rect(0, GRID_HEIGHT, WINDOW_WIDTH, BOTTOM_PICKER_HEIGHT)
    pygame.draw.rect(screen, BOTTOM_BAR_COLOR, bottom_rect)
    pygame.draw.line(screen, BORDER_COLOR, (0, GRID_HEIGHT), (WINDOW_WIDTH, GRID_HEIGHT), 2)

    title = font.render("Color Picker", True, TEXT_COLOR)
    screen.blit(title, (20, GRID_HEIGHT + 12))

    helper_text = "Pick target first. The color palette opens as a popup."
    helper_color = MUTED_TEXT
    helper = small_font.render(helper_text, True, helper_color)
    screen.blit(helper, (20, GRID_HEIGHT + 38))

    if picker_message:
        alert_rect = pygame.Rect(WINDOW_WIDTH - 510, GRID_HEIGHT + 12, 490, 30)
        pygame.draw.rect(screen, ALERT_BG, alert_rect, border_radius=8)
        pygame.draw.rect(screen, ALERT_COLOR, alert_rect, 2, border_radius=8)

        alert_text = small_font.render(picker_message, True, ALERT_COLOR)
        screen.blit(alert_text, (alert_rect.x + 12, alert_rect.y + 7))

    target_rects, swatch_rects, editing_rect, popup_rect = get_color_picker_layout()
    selected_color = COLOR_MAP.get(selected_target, COLOR_FALLBACK)
    target_label = next(label for target, label in COLOR_TARGETS if target == selected_target)

    pygame.draw.rect(screen, (42, 48, 61), editing_rect, border_radius=10)
    pygame.draw.rect(screen, HIGHLIGHT, editing_rect, 3, border_radius=10)
    pygame.draw.rect(screen, selected_color, pygame.Rect(editing_rect.x, editing_rect.y, 8, editing_rect.h), border_radius=6)

    preview_color_rect = pygame.Rect(editing_rect.x + 18, editing_rect.y + 13, 30, 22)
    pygame.draw.rect(screen, selected_color, preview_color_rect, border_radius=5)
    pygame.draw.rect(screen, BORDER_COLOR, preview_color_rect, 1, border_radius=5)

    editing_title = small_font.render("EDITING", True, HIGHLIGHT)
    editing_value = font.render(target_label, True, TEXT_COLOR)
    screen.blit(editing_title, (editing_rect.x + 58, editing_rect.y + 6))
    screen.blit(editing_value, (editing_rect.x + 58, editing_rect.y + 24))

    for rect, target, label in target_rects:
        is_active = target == selected_target
        color = BUTTON_ACTIVE if is_active else BUTTON_COLOR
        pygame.draw.rect(screen, color, rect, border_radius=6)
        pygame.draw.rect(screen, HIGHLIGHT if is_active else BORDER_COLOR, rect, 2, border_radius=6)

        label_surface = small_font.render(label, True, TEXT_COLOR)
        label_pos = label_surface.get_rect(center=rect.center)
        screen.blit(label_surface, label_pos)

    if palette_open:
        pygame.draw.rect(screen, (38, 43, 55), popup_rect, border_radius=10)
        pygame.draw.rect(screen, HIGHLIGHT, popup_rect, 2, border_radius=10)

        for rect, name, color in swatch_rects:
            pygame.draw.rect(screen, color, rect, border_radius=5)
            border_width = 3 if color == selected_color else 1
            border_color = HIGHLIGHT if color == selected_color else BORDER_COLOR
            pygame.draw.rect(screen, border_color, rect, border_width, border_radius=5)





def find_cell(grid, value):
    for row in range(ROWS):
        for col in range(COLS):
            if grid[row][col] == value:
                return row, col
    return None


def bfs_path_generator(grid, start_cell, end_cell):
    queue = deque([start_cell])
    visited = {start_cell}
    parent = {}

    while queue:
        current_row, current_col = queue.popleft()

        if (current_row, current_col) == end_cell:
            path = []
            current = end_cell

            while current != start_cell:
                path.append(current)
                current = parent[current]

            path.reverse()

            for row, col in path:
                yield ("path", row, col)

            return

        neighbors = [
            (current_row - 1, current_col),
            (current_row + 1, current_col),
            (current_row, current_col - 1),
            (current_row, current_col + 1),
        ]

        for row, col in neighbors:
            if not is_valid_cell(grid, row, col):
                continue

            value = get_cell(grid, row, col)

            if value in (WALL, VISITED):
                continue

            if (row, col) in visited:
                continue

            visited.add((row, col))
            parent[(row, col)] = (current_row, current_col)
            queue.append((row, col))
            yield ("visit", row, col)

    yield ("no_path",)





def run_color_picker_ui_demo():
    global COLOR_MAP

    pygame.init()
    screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
    pygame.display.set_caption("Color Picker Demo")

    clock = pygame.time.Clock()
    font = pygame.font.SysFont("consolas", 20, bold=True)
    small_font = pygame.font.SysFont("consolas", 16)

    grid = create_grid(ROWS, COLS)
    mode = MODE_WALL
    selected_target = FILLED
    palette_open = False
    picker_message = ""

    start_cell = None
    end_cell = None
    bfs_running = False
    bfs_generator = None
    bfs_timer = 0
    running = True

    while running:
        dt = clock.tick(FPS)
        target_rects, swatch_rects, editing_rect, popup_rect = get_color_picker_layout()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                elif event.key == pygame.K_w:
                    mode = MODE_WALL
                elif event.key == pygame.K_f:
                    mode = MODE_FILL
                elif event.key == pygame.K_e:
                    mode = MODE_ERASE
                elif event.key == pygame.K_s:
                    mode = MODE_START
                elif event.key == pygame.K_d:
                    mode = MODE_END
                elif event.key == pygame.K_r:
                    grid = create_grid(ROWS, COLS)
                    start_cell = None
                    end_cell = None
                    bfs_running = False
                    bfs_generator = None
                    palette_open = False
                    picker_message = ""
                elif event.key == pygame.K_c:
                    COLOR_MAP = DEFAULT_COLOR_MAP.copy()
                    palette_open = False
                    picker_message = ""
                elif event.key == pygame.K_SPACE:
                    start_cell = find_cell(grid, START)
                    end_cell = find_cell(grid, END)

                    if start_cell and end_cell:
                        for row in range(ROWS):
                            for col in range(COLS):
                                if get_cell(grid, row, col) in (VISITED, PATH):
                                    set_cell(grid, row, col, EMPTY)

                        bfs_generator = bfs_path_generator(grid, start_cell, end_cell)
                        bfs_running = True
                        bfs_timer = 0
                        palette_open = False
                        picker_message = ""

            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                clicked_ui = False

                if palette_open:
                    for rect, name, color in swatch_rects:
                        if rect.collidepoint(event.pos):
                            if color_choice_is_blocked(selected_target, color):
                                picker_message = "Path and Wall cannot share a color. Choose again."
                                palette_open = True
                            else:
                                COLOR_MAP[selected_target] = color
                                picker_message = ""
                                palette_open = False
                            clicked_ui = True

                    if not clicked_ui and popup_rect.collidepoint(event.pos):
                        clicked_ui = True

                if not clicked_ui and event.pos[1] >= GRID_HEIGHT:
                    clicked_ui = True
                    target_clicked = False

                    for rect, target, label in target_rects:
                        if rect.collidepoint(event.pos):
                            selected_target = target
                            palette_open = True
                            picker_message = ""
                            target_clicked = True

                    if not target_clicked:
                        palette_open = False
                        picker_message = ""

                elif not clicked_ui and event.pos[0] >= GRID_WIDTH:
                    clicked_ui = True
                    palette_open = False
                    picker_message = ""

                elif palette_open:
                    palette_open = False
                    picker_message = ""
                    clicked_ui = True

                if clicked_ui or bfs_running:
                    continue

                clicked_cell = mouse_to_grid_cell(event.pos)

                if clicked_cell is None:
                    continue

                row, col = clicked_cell

                if mode == MODE_WALL:
                    if get_cell(grid, row, col) not in (START, END):
                        set_cell(grid, row, col, WALL)

                elif mode == MODE_FILL:
                    if get_cell(grid, row, col) not in (START, END):
                        set_cell(grid, row, col, FILLED)

                elif mode == MODE_ERASE:
                    if start_cell == (row, col):
                        start_cell = None
                    if end_cell == (row, col):
                        end_cell = None
                    set_cell(grid, row, col, EMPTY)

                elif mode == MODE_START:
                    old_start = find_cell(grid, START)
                    if old_start:
                        set_cell(grid, old_start[0], old_start[1], EMPTY)

                    start_cell = (row, col)
                    set_cell(grid, row, col, START)

                elif mode == MODE_END:
                    old_end = find_cell(grid, END)
                    if old_end:
                        set_cell(grid, old_end[0], old_end[1], EMPTY)

                    end_cell = (row, col)
                    set_cell(grid, row, col, END)

        if bfs_running and bfs_generator is not None:
            bfs_timer += dt

            if bfs_timer >= BFS_DELAY:
                bfs_timer = 0

                try:
                    step = next(bfs_generator)

                    if step[0] == "visit":
                        _, row, col = step

                        if get_cell(grid, row, col) in (EMPTY, FILLED):
                            set_cell(grid, row, col, VISITED)

                    elif step[0] == "path":
                        _, row, col = step

                        if get_cell(grid, row, col) not in (START, END):
                            set_cell(grid, row, col, PATH)

                    elif step[0] == "no_path":
                        print("No path found")
                        bfs_running = False

                except StopIteration:
                    bfs_running = False

        screen.fill(COLOR_GRID_BG)
        draw_grid(screen, grid)
        draw_panel(screen, font, small_font, mode)
        draw_color_picker(screen, font, small_font, selected_target, palette_open, picker_message)
        pygame.display.flip()

    pygame.quit()


run_color_picker_ui_demo()


: 